# pandas for financial statements

**Before you run anything:** keep `statement.csv` in the *same folder* as this notebook.

Run the cells top to bottom.

Last lesson you loaded `statement.csv` by hand: a `csv.DictReader` loop, a `parse_amount` function, a `parse_date` function, a `defaultdict` for grouping, an `OrderedDict` walk for reconciliation, a `seen` set for dedup. Every one of those was you building, one piece at a time, the exact thing `pandas` gives you out of the box: a table of typed columns you can slice, group, and aggregate directly. This lesson redoes the same five exercises, now the pandas way, so you can see line-for-line what the library buys you.

## 1. Load the statement into a DataFrame

`pd.read_csv` reads the whole file into a `DataFrame`, a table where every column has its own type. Compare `.head()` here to the manual `rows[:3]` loop from last lesson: same information, no loop required.

In [ ]:
import pandas as pd

df = pd.read_csv("statement.csv")
print("shape:", df.shape)
df.head(3)

`.dtypes` shows what pandas guessed for each column. `Amount` and `Daily Posted Balance` come in as `object` (plain strings), because of the `$`, commas, and `($86.32)`-style parentheses, pandas can't tell those are numbers yet. `Posted Date` and `Transaction Date` are also `object`, still just text. This is the same problem `parse_amount` and `parse_date` solved by hand; here we solve it once, for the whole column at a time.

In [ ]:
df.dtypes

## 2. Parse amounts and dates, a column at a time

No per-row function needed. `.str` gives you vectorized string operations that run on every row of a column at once:

- strip `$` and commas with `.str.replace`
- detect the parentheses with `.str.startswith` / `.str.endswith` to build a boolean mask of debits
- strip the parentheses, convert to `float`, then flip the sign where the mask is `True`

`pd.to_datetime` replaces `parse_date` for both date columns in one call each.

In [ ]:
def parse_amount_column(col):
    is_debit = col.str.strip().str.startswith("(") & col.str.strip().str.endswith(")")
    cleaned = col.str.strip().str.strip("()").str.replace("$", "", regex=False).str.replace(",", "", regex=False)
    values = cleaned.astype(float)
    return values.where(~is_debit, -values)

df["amount"] = parse_amount_column(df["Amount"])
df["daily_balance"] = parse_amount_column(df["Daily Posted Balance"])
df["posted_date"] = pd.to_datetime(df["Posted Date"], format="%m/%d/%Y")
df["txn_date"] = pd.to_datetime(df["Transaction Date"], format="%m/%d/%Y")

# sanity checks, same values as the manual parse_amount assertions last lesson
assert df.loc[df["Full description"].str.startswith("FRESH MARKET 4821 06-02"), "amount"].iloc[0] == -86.32
df[["Posted Date", "posted_date", "Amount", "amount"]].head(3)

## Exercise 1 — Confirm the sign convention

Same check as last lesson: debits negative, credits positive, and the net change over the month. Boolean indexing (`df[df["amount"] < 0]`) replaces the manual list comprehension over `records`.

In [ ]:
debits = df[df["amount"] < 0]
credits = df[df["amount"] > 0]
print("debits :", len(debits), "  total", round(debits["amount"].sum(), 2))
print("credits:", len(credits), "  total", round(credits["amount"].sum(), 2))
print("net change over the month:", round(df["amount"].sum(), 2))

## Exercise 2 — Spending by category

This is the one line that replaces the `defaultdict` loop entirely: `.groupby(...)["amount"].sum()`. As before, we use `txn_date`-based rows implicitly (spend is just the negative amounts here, regardless of which date column, but recall from last lesson: whenever the *question* is about when money was spent, group or filter on `txn_date`, not `posted_date`).

In [ ]:
spend = (
    df[df["amount"] < 0]
    .assign(spend=lambda d: -d["amount"])
    .groupby("Category name")["spend"]
    .sum()
    .sort_values(ascending=False)
)
spend

## Exercise 3 — Reconcile a day's balance

Same idea as the manual `OrderedDict` walk: each posted date's closing balance should equal the previous posted date's closing balance plus that day's transactions. `.groupby("Posted Date")` gets us the daily total and the closing balance in two aggregations; `.diff()` lines up each day against the one before it without an explicit `prev` variable.

In [ ]:
daily = df.groupby("posted_date").agg(
    day_total=("amount", "sum"),
    closing=("daily_balance", "last"),
).sort_index()

daily["expected"] = daily["closing"].shift(1) + daily["day_total"]
daily["expected"] = daily["expected"].round(2)
daily["flag"] = "OK"
daily.loc[(daily["expected"] - daily["closing"]).abs() >= 0.005, "flag"] = "MISMATCH"
daily.loc[daily.index[0], "flag"] = "(first day)"
daily

## Exercise 4 — How often does the merchant fallback work?

Where `.str` vectorization can't reach, `.apply()` is the escape hatch: it calls a plain Python function once per row, same as a `for` loop would, just wrapped in a familiar interface. It's not vectorized and it's slower than the operations above, so reach for it only when the logic genuinely doesn't reduce to string/boolean ops, exactly what's true here since `extract_merchant` is a regex written for one string at a time.

In [ ]:
import re

MERCHANT_PATTERN = re.compile(r"([A-Z][A-Za-z&' ]+?)\s+\d")

def extract_merchant(description):
    m = MERCHANT_PATTERN.match(description)
    return m.group(1).strip() if m else description

blanks = df[df["Merchant name"].isna()].copy()
blanks["guess"] = blanks["Full description"].apply(extract_merchant)
print(len(blanks), "rows had no merchant name\n")
blanks[["Full description", "guess"]]

`.fillna` takes it the rest of the way: everywhere `Merchant name` is blank, fall back to the regex guess; everywhere it isn't, keep the bank's own value. This is the `.apply()` output from above folded straight into the column, no separate `or extract_merchant(...)` per row.

In [ ]:
df["merchant"] = df["Merchant name"].fillna(df["Full description"].apply(extract_merchant))
df[["Merchant name", "merchant"]].head(3)

## Exercise 5 — The near-duplicate trap, revisited

`.duplicated(subset=[...])` is pandas' built-in version of the `seen`-set loop from last lesson, and it has the exact same blind spot: keying only on `(txn_date, merchant, category)` still can't tell apart the two coffees on 06/10 or the two rideshares on 06/14, they're real, distinct transactions that only differ by amount. A one-line call doesn't fix a lossy key; it just makes it faster to be wrong.

In [ ]:
dupe_mask = df.duplicated(subset=["txn_date", "merchant", "Category name"], keep="first")
print("flagged as duplicates:", dupe_mask.sum())
df.loc[dupe_mask, ["txn_date", "merchant", "Full description", "amount"]]

## 3. One chart: spend by category

`DataFrame.plot` is a thin wrapper over matplotlib. Passing a `Series` like `spend` from Exercise 2 straight to `.plot(kind="bar")` is usually enough for a first look, no separate charting code needed for something this simple.

In [ ]:
spend.plot(kind="bar", title="Spending by category", ylabel="dollars")

## Where this points next

You've now solved the same five problems two ways: by hand, then with pandas. The library didn't change *what* the questions were, sign checks, grouping, reconciliation, fallback recovery, dedup, it changed how much code it took to answer them. Real financial analysis usually spans more than one statement: the natural next step is combining several months of statements into one DataFrame and tracking spending trends over time.